In [2]:
%mamba install pandas

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 3.006 seconds
  Name           Version  Build                Channel
--------------------------------------------------------------------
+ pandas         3.1.0    np23py313h124dbc7_0  emscripten-forge-4x
+ python-tzdata  2026.5   pyh5ded981_0         conda-forge
- pip            26.2.1   pyh145f28c_0         conda-forge


In [3]:
%mamba install scikit-learn

mambajs 0.21.4

Specs: xeus-python, numpy, matplotlib, pillow, ipywidgets>=8.1.6, ipyleaflet, scipy, pandas, scikit-learn
Channels: emscripten-forge-4x, conda-forge

Solving environment...
Solving took 1.935 seconds
  Name                Version     Build                Channel
----------------------------------------------------------------------------
+ brotli-python       1.2.0       py313h8faef1e_3      emscripten-forge-4x
+ certifi             2026.10.23  pyhd8ed1ab_0         conda-forge
+ charset-normalizer  3.5.1       pyhd8ed1ab_0         conda-forge
+ idna                3.20        pyh5ded981_0         conda-forge
+ joblib              1.6.0       py313h57e9779_0      emscripten-forge-4x
+ narwhals            2.27.1      pyh5ded981_0         conda-forge
+ pysocks             1.7.1       py313h1804a44_3      emscripten-forge-4x
+ requests            2.34.2      pyhcf101f3_0         conda-forge
+ scikit-learn        1.9.0       np23py313hb9816c2_0  emscripten-forge-4x
+ setupto

In [5]:
# Loading the dataset and preparing the inputs

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV, ElasticNetCV
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

# Load dataset
df = pd.read_csv("D_motor_thermal.csv")

# Separate training and test rows using the given split
train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()

# Select all 30 input features
feature_cols = [
    col for col in df.columns
    if col not in ["split", "winding_rise_K"]
]

X_train = train_df[feature_cols]
y_train = train_df["winding_rise_K"]

X_test = test_df[feature_cols]
y_test = test_df["winding_rise_K"]

# Required cross-validation splitter: Rule R5
cv = KFold(n_splits=5, shuffle=True, random_state=42)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))
print("Number of input features:", len(feature_cols))

Training rows: 60
Testing rows: 500
Number of input features: 30


In [6]:
# D1 - Ordinary Least Squares (OLS)

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Fit OLS using standardized inputs
ols_model = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

ols_model.fit(X_train, y_train)

# Predictions
y_train_pred = ols_model.predict(X_train)
y_test_pred = ols_model.predict(X_test)

# RMSE
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"Training RMSE: {train_rmse:.3f} K")
print(f"Test RMSE: {test_rmse:.3f} K")

# Extract coefficients and match them with feature names
lr = ols_model.named_steps["linearregression"]

coef_table = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": lr.coef_
})

coef_table["Absolute coefficient"] = coef_table["Coefficient"].abs()
top_five = coef_table.nlargest(5, "Absolute coefficient")

display(top_five[["Feature", "Coefficient"]].round(2))

Training RMSE: 1.252 K
Test RMSE: 2.915 K


,Feature,Coefficient
21,torque_Nm,5.66
4,i_band_05,3.26
24,coolant_flow_lpm,-3.03
6,i_band_07,1.56
10,i_band_11,0.98


In [10]:
# D2 - Ridge, Lasso and Elastic Net

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

cv = KFold(n_splits=5, shuffle=True, random_state=42)
alphas = np.logspace(-3, 3, 61)

estimators = {
    "Ridge": RidgeCV(alphas=alphas, cv=cv),
    "Lasso": LassoCV(
        cv=cv, max_iter=50000, random_state=0
    ),
    "Elastic Net": ElasticNetCV(
        l1_ratio=[0.2, 0.5, 0.8],
        cv=cv, max_iter=50000, random_state=0
    )
}

fitted_models = {}

for name, estimator in estimators.items():
    model = make_pipeline(StandardScaler(), estimator)
    model.fit(X_train, y_train)
    fitted_models[name] = model

    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    fitted_estimator = model.steps[-1][1]

    print(f"\n{name}")
    print(f"Selected alpha: {fitted_estimator.alpha_:.3g}")
    print(f"Test RMSE: {rmse:.3f} K")

    if name == "Elastic Net":
        print("Selected l1_ratio:", fitted_estimator.l1_ratio_)

    coefficients = pd.Series(
        fitted_estimator.coef_,
        index=X_train.columns,
        name="Coefficient"
    )

    nonzero_coefficients = coefficients[coefficients != 0]

    print("Number of non-zero coefficients:",
          len(nonzero_coefficients))


Ridge
Selected alpha: 0.398
Test RMSE: 2.890 K
Number of non-zero coefficients: 30

Lasso
Selected alpha: 0.105
Test RMSE: 2.413 K
Number of non-zero coefficients: 21

Elastic Net
Selected alpha: 0.107
Test RMSE: 2.506 K
Selected l1_ratio: 0.8
Number of non-zero coefficients: 23


In [14]:
# D3 - Lasso feature selection stability

import numpy as np
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.model_selection import KFold

# Required 5-fold cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Features specifically listed in the D3 assignment table
required_features = [
    "torque_Nm",
    "i_band_01",
    "i_band_05",
    "i_band_06",
    "i_band_07",
    "i_band_17",
    "coolant_flow_lpm",
    "ambient_C",
    "aux_02",
    "aux_05"
]

# Check that the required features exist
missing = [
    f for f in required_features
    if f not in X_train.columns
]

if missing:
    raise ValueError(f"Feature names not found: {missing}")

# Count selections only for the 10 requested features
selection_counts = pd.Series(
    0, index=required_features, dtype=int
)

# Store total features selected in each fit
features_per_fit = []

# Run LassoCV for seeds 0 to 9
for s in range(10):

    # Sample 60 training indices with replacement
    idx = np.random.default_rng(s).integers(0, 60, 60)

    X_sample = X_train.iloc[idx]
    y_sample = y_train.iloc[idx]

    # Fit scaler and LassoCV
    model = make_pipeline(
        StandardScaler(),
        LassoCV(
            cv=cv,
            max_iter=50000,
            random_state=0
        )
    )

    model.fit(X_sample, y_sample)

    # Extract coefficients for all 30 features
    coefficients = pd.Series(
        model.named_steps["lassocv"].coef_,
        index=X_train.columns
    )

    # Identify all non-zero coefficients
    selected = coefficients[coefficients != 0].index

    # Count selections for the 10 features in the assignment table
    for feature in required_features:
        if feature in selected:
            selection_counts[feature] += 1

    # Count total selected features across all 30 inputs
    features_per_fit.append(len(selected))

# Display only the required D3 table
d3_table = pd.DataFrame({
    "Feature": required_features,
    "Selected (/10)": selection_counts.values
})

print("\nD3: Selection Stability")
print(d3_table.to_string(index=False))

# Required minimum and maximum
print(
    "\nSmallest number of features selected:",
    min(features_per_fit)
)

print(
    "Largest number of features selected:",
    max(features_per_fit)
)


D3: Selection Stability
         Feature  Selected (/10)
       torque_Nm              10
       i_band_01               1
       i_band_05              10
       i_band_06               8
       i_band_07              10
       i_band_17              10
coolant_flow_lpm              10
       ambient_C               8
          aux_02              10
          aux_05              10

Smallest number of features selected: 15
Largest number of features selected: 26
